# How local is local?
Recreate the article’s bond and option comparisons using the pricing functions and experiment grids in this checkout. Run from the repository root or the notebooks directory. Currency figures are per £1,000 face for bonds and per share for calls.

In [ ]:
from pathlib import Path
import sys

root = Path.cwd()
if root.name == "notebooks":
    root = root.parent
sys.path[:0] = [str(root / "src"), str(root)]

from research.sweep import bond_rows, call_rows, crossings, scaled_calls

def show(rows, fields):
    print("  ".join(f"{field:>16}" for field in fields))
    for row in rows:
        values = [row[field] for field in fields]
        print("  ".join(f"{v:16.4f}" if isinstance(v, float) else f"{str(v):>16}" for v in values))

## The same bond at three yields
Change the maturity or select different shocks. The linear and quadratic estimates use sensitivities measured at 4%.

In [ ]:
years = 5
rows = list(bond_rows(years))
show([r for r in rows if r["shock"] in (-100, 0, 100)],
     ["shock", "initial_price", "exact", "linear", "second"])

## When does the miss matter?
Walk outwards from zero on each side. A blank crossing (None) means the estimate stayed within budget across that side of the grid.

In [ ]:
show([r for r in crossings(rows) if r["tolerance_pct"] == 1],
     ["direction", "approximation", "first_sample_above"])

## Keep the shock fixed and change expiry
Set a spot shock in percentage points. The seven-day call is still seven days from expiry after the instantaneous shock.

In [ ]:
spot_shock = -10
sample = []
for name, years in [("one year", 1), ("30 days", 30/365), ("seven days", 7/365)]:
    sample.extend(call_rows(name, years, shocks=[spot_shock]))
show(sample, ["case", "shock", "exact", "linear", "second"])

## Keep the volatility multiple fixed instead
At each expiry apply a spot fall of half the remaining volatility scale. Compare the quadratic error as a percentage of absolute repriced P&L.

In [ ]:
show([r for r in scaled_calls() if r["vol_scale_multiple"] == -0.5],
     ["case", "shock", "exact", "second", "second_error_pct"])

## Published research figures
The tracked figures below come from the full scenario grid. Regenerate them from the repository root with `make figures`.

![Bond and call P&L and errors](../outputs/bond-and-call.png)

![Expiry comparison](../outputs/expiry-stress.png)